In [1]:
import sys
sys.path.insert(0, '..')

from helper.golbal_variables import * 
from ast_dataset import Fast_AST_Dataset, fast_collate_fn
from ast_model import ASTGenreClassifier
from cross_validation_pipeline import Pipeline

z:\dlgen-AI-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
modle_pipeline = Pipeline(dataset_path=GENRES_PT)
modle_pipeline.cross_validation()

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [ ]:
import torch 
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 10
model = ASTGenreClassifier(num_classes)
model.load_state_dict(torch.load("best_ast_genre_s3_v3.pth", map_location=device))
model = model.to(device)

z:\dlgen-AI-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\visha\AppData\Local\Temp\ipykernel_13612\792667690.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default val

In [ ]:
import pandas as pd
from predict import predict_ast

genre_map = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}

test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv")
output = {}
for idx, row in test_df.iterrows():
    song_id = row["id"]
    filename = row["filename"]
    audio_path = MESSY_DATA_DIR / filename

    predicted_genre, confidence, all_probs = predict_ast(
        model=model,
        audio_path=audio_path,
        device=device,
        genre_map=genre_map
    )

    output[song_id] = predicted_genre
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} songs...")

output_df = pd.DataFrame(list(output.items()), columns=['id', 'genre'])
output_df.set_index("id", inplace=True)
output_df.to_csv("output 5.csv")